# Задание 1.
Реализуйте базовый класс Account, который моделирует поведение
банковского счёта. Этот класс должен не только выполнять базовые
операции, но и вести детальный учёт всех действий, а также предоставлять
аналитику по истории операций.

## Этап 1. Реализация базового класса Account
Класс должен быть инициализирован с параметрами:
* `account_holder (str)` — имя владельца счёта;
* `balance (float, по умолчанию 0)` — начальный баланс счёта, не может быть отрицательным.

### Атрибуты:
* `_account_counter` — приватный атрибут для хранения количества созданных счетов. Отсчет начинается с 1000;
* `holder` — хранит имя владельца;
* `account_number` — хранит номер счёта;
* `_balance` — приватный атрибут для хранения текущего баланса;
* `operations_history` — список или другая структура для хранения истории операций.

Важно: каждая операция должна храниться не просто как число, а как
структурированная информация, например, словарь, кортеж или класс.
Минимальный набор данных для операции: тип операции (`'deposit'` или
`'withdraw'`), сумма, дата и время операции, текущий баланс после операции,
статус (`'success'` или `'fail'`).

---

## Этап 2. Реализация методов
1. `__init__(self, account_holder, balance=0)` — конструктор. Обратите
внимание, что в конструкторе должен автоматически формироваться
номер счёта в формате `‘ACC-XXXX’`, где `XXXX` — порядковый номер
счёта;
2. `deposit(self, amount)` — метод для пополнения счёта:
    * принимает сумму (должна быть положительной), попытка
положить отрицательную сумму, должна вызывать исключение;
    * в случае успеха обновляет баланс и добавляет запись в историю операций.
3. `withdraw(self, amount)` — метод для снятия средств:
    * принимает сумму (должна быть положительной);
    * проверяет, достаточно ли средств на счёте, если нет — операция
не проходит, но её попытка со статусом `'fail'` всё равно фиксируется
в истории;
    * в случае успеха обновляет баланс и добавляет запись в историю.
4. `get_balance(self)` — метод, который возвращает текущий баланс.
5. `get_history(self)` — метод, который возвращает историю операций.

Важно: продумайте, в каком формате его вернуть. Для работы с датой и
временем используйте модуль `datetime`. Получить текущее время можно с
помощью `datetime.now()`.

---

## Этап 3. Визуализация истории операций. Дополнительное задание для претендующих на оценку 8 и выше баллов (выполняется по желанию).
1. Создайте метод `plot_history(self)`, который использует библиотеку
Pandas для создания датафрейма из истории операций.
2. Продумайте, с помощью какой библиотеки можно отобразить
изменение баланса с течением времени. Постройте простой
линейный график, где по оси X будет время операции, а по оси Y —
баланс после каждой операции. График должен иметь заголовок,
подписи осей.


In [ ]:
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import re

class Operation:
    def __init__(self, operation_type, amount, balance_after, status):
        self.operation_type = operation_type
        self.amount = amount
        self.time = datetime.now()
        self.balance_after = balance_after
        self.status = status

    def __repr__(self):
        return f"{self.time:%Y-%m-%d %H:%M:%S} | {self.operation_type} {self.amount} | Баланс: {self.balance_after} | {self.status}"


class Account ():
    _account_counter = 1000

    @staticmethod
    def _validate_holder(name: str):
        pattern = r'^(?:[A-Z][a-z]+ [A-Z][a-z]+|[А-ЯЁ][а-яё]+ [А-ЯЁ][а-яё]+)$'
        if not re.fullmatch(pattern, name):
            raise ValueError("Имя владельца должно быть в формате 'Имя Фамилия' с заглавных букв (латиница или кириллица).")

    def __init__(self, account_holder, balance = 0):
        if balance < 0:
            raise ValueError("Баланс не может может быть отрицательным")
        
        Account._validate_holder(account_holder)
        
        self.holder = account_holder
        self._balance = balance

        Account._account_counter += 1

        self.account_number = f'ACC-{Account._account_counter}' 

        self.operations_history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('Сумма депозита должна быть положительной')
        else:
            self._balance += amount
            self.operations_history.append(Operation('deposit', amount, self._balance, 'success'))

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError('Сумма снятия должна быть положительной')
        
        if self._balance - amount < 0:
            self.operations_history.append(Operation('withdraw', amount, self._balance, 'fail'))
        else:
            self._balance -= amount
            self.operations_history.append(Operation('withdraw', amount, self._balance, 'success'))
    
    def get_balance(self):
        return self._balance
    
    def get_history(self):
        return self.operations_history
    
    def plot_history(self):
        if not self.operations_history:
            print("История пуста, нечего визуализировать.")
            return

        df = pd.DataFrame([{
            'Время': op.time,
            'Тип операции': op.operation_type,
            'Сумма': op.amount,
            'Баланс после операции': op.balance_after,
            'Статус': op.status
        } for op in self.operations_history])

        plt.plot(df['Время'], df['Баланс после операции'], marker='o')
        plt.title(f'Изменение баланса счёта {self.account_number}')
        plt.xlabel('Время операции')
        plt.ylabel('Баланс (₽)')
        plt.grid(True)
        plt.show()


    def recent_operations(self, n = 5, min_amount: float = 100000, only_success: bool = True):
        ops = self.operations_history

        if only_success:
            ops = [op for op in ops if op.status == 'success']

        ops = [op for op in ops if abs(op.amount) >= min_amount]

        ops.sort(key=lambda op: op.time, reverse=True)

        return ops[:n]
    

class SavingsAccount(Account):
    account_type = 'Сберегательный счёт'
    
    def apply_interest(self, rate):
        if rate < 0:
            raise ValueError("Ставка не может быть отрицательной")
        
        interest = self._balance * (rate / 100)
        self._balance += interest
        self.operations_history.append(Operation('apply_interest', interest, self._balance, 'success'))

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError('Сумма снятия должна быть положительной')
        
        if amount > self._balance * 0.5:
            self.operations_history.append(Operation('withdraw', amount, self._balance, 'fail'))
        else:
            self._balance -= amount
            self.operations_history.append(Operation('withdraw', amount, self._balance, 'success'))



class CheckingAccount(Account):
    account_type = 'Расчётный счёт'
